<a href="https://colab.research.google.com/github/acapodanno/openai-agent-sdk/blob/main/nick_distructor_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pydantic

In [ ]:
from pydantic import BaseModel,Field
class Article(BaseModel):
  title: str = Field(description="Title of the article")
  content: str  = Field(description="Content of the article. It should contain html tags")

In [ ]:
!pip install openai-agents

In [ ]:
from agents import Agent, Runner,set_default_openai_key

#### Define Agent
async def run_agent(project_name):
  agent = Agent(
      name="Web3 Article Generator",

      instructions="You are a senior crypto/web3 expert. "
          "Generate structured HTML articles about web3 projects.",
  model="gpt-5-mini",
  output_type=Article
  )

  runner = Runner()
  return await runner.run(agent,f"""
Write a detailed, well‑structured article about the web3 project "{project_name}".
Write in English.

Structure:
- Catchy title
- Introduction
- What the project is
- Technology and architecture
- Tokenomics (if applicable)
- Use cases
- Roadmap
- Risks
- Conclusion

Return HTML (<h2>, <p>, <ul>, <li>). Formart for Word Press Elementor
""")

In [ ]:
OPENAI_API_KEY = "" #ADD OPENAI KEY
set_default_openai_key(OPENAI_API_KEY)



In [ ]:
from requests import post
WP_BASE_URL = "https://tuosito.com"
WP_USERNAME = "admin"
WP_APP_PASSWORD = "xxxx xxxx xxxx xxxx"

def public_draft_wp(titolo: str, contenuto_html: str) -> dict:
    endpoint = f"{WP_BASE_URL.rstrip('/')}/wp-json/wp/v2/posts"
    auth = (WP_USERNAME, WP_APP_PASSWORD)

    payload = {
        "title": titolo,
        "content": contenuto_html,
        "status": "draft"
    }
    response = post(endpoint, json=payload, auth=auth)

    if not response.ok:
        print("Errore WordPress:", response.status_code, response.text)
        response.raise_for_status()

    return response.json()


In [ ]:
async def main():
  project_name = input("Enter name project: ")
  result = await run_agent(project_name)
  article = result.final_output
  print(article)
  public_draft_wp(article.title,article.content)
await main()